In [ ]:
import os
import sys
sys.path.append('./coeqwalpackage')

import numpy as np
import pandas as pd
import datetime as dt

from coeqwalpackage.metrics import read_in_df, add_water_year_column, probability_var1_gte_var2_for_scenario, probability_var1_gte_const_for_scenario
from coeqwalpackage.tier import classify_flood_tier
import cqwlutils as cu

## Configuration

In [ ]:
# Configuration parameters
CtrlFile = 'CalSim3DataExtractionInitFile_v4.xlsx'
CtrlTab = 'Init'

# Tier thresholds
tier1_tier2_thresh = 0.1
tier2_tier3_thresh = 0.4

# Variables
variables_storage = ["S_SHSTA_", "S_OROVL_", "S_FOLSM_", "S_TRNTY_", "S_SLUIS_CVP_", "S_SLUIS_SWP_", "S_MLRTN_", "S_MELON_"]
variables_floodpool = ["S_SHSTALEVEL5DV", "S_OROVLLEVEL5DV", "S_FOLSMLEVEL5DV", "S_TRNTYLEVEL4DV", "S_SLUIS_CVPLEVEL4DV", "S_SLUIS_SWPLEVEL4DV", "S_MLRTNLEVEL4DV", "S_MELONLEVEL4DV"]

# Constants
S_MLRTN_level5constant = 524

# Output file names
tier_output_filename = "tier_assignment.csv"
prob_output_filename = "probabilities_assignment.csv"

## Read From Control File

In [ ]:
ScenarioListFile, ScenarioListTab, ScenarioListPath, DVDssNamesOutPath, SVDssNamesOutPath, ScenarioIndicesOutPath, DssDirsOutPath, VarListPath, VarListFile, VarListTab, VarOutPath, DataOutPath, ConvertDataOutPath, ExtractionSubPath, DemandDeliverySubPath, ModelSubPath, GroupDataDirPath, ScenarioDir, DVDssMin, DVDssMax, SVDssMin, SVDssMax, NameMin, NameMax, DirMin, DirMax, IndexMin, IndexMax, StartMin, StartMax, EndMin, EndMax, VarMin, VarMax, DemandFilePath, DemandFileName, DemandFileTab, DemMin, DemMax, InflowOutSubPath, InflowFilePath, InflowFileName, InflowFileTab, InflowMin, InflowMax = cu.read_init_file(CtrlFile, CtrlTab)

## Paths

In [ ]:
tiers_output_dir = os.path.join(ScenarioDir, "Performance_Metrics", "Tiered_Outcome_Measures", "Reservoir_FloodRisk", "Tiers")
tiers_output_path = os.path.join(tiers_output_dir, tier_output_filename)
probs_output_dir = os.path.join(ScenarioDir, "Performance_Metrics", "Metrics", "Reservoir_FloodRisk")
probs_output_path = os.path.join(probs_output_dir, prob_output_filename)

if not os.path.exists(tiers_output_dir):
    print("Warning: directory " + tiers_output_dir + " does not exist and will be created")
    os.makedirs(tiers_output_dir)

if not os.path.exists(probs_output_dir):
    print("Warning: directory " + probs_output_dir + " does not exist and will be created")
    os.makedirs(probs_output_dir)

## Read in Data

In [ ]:
df, dss_names = read_in_df(ConvertDataOutPath, DVDssNamesOutPath)
df = add_water_year_column(df)

## Tier Assignments

#### Compute Probabilities and Tiers

In [ ]:
# hitting floodpool = variables_storage >= variables_floodpool
# FOLSOM is NaN for scenario 6-10 because it doesn't have level5D for scenario 6-10
S_MLRTN_level5constant = 524

probs_result_list = []
tiers_result_list = []

for scenario in dss_names:
    scenario_short = scenario[:5]
    probs_row_data = {"Scenario": scenario_short}
    tiers_row_data = {"Scenario": scenario_short}

    for var_s, var_lvl5D in zip(variables_storage, variables_floodpool):
        var1_name = f"{var_s}{scenario_short}"
        var2_name = f"{var_lvl5D}_{scenario_short}"

        p_gte = probability_var1_gte_var2_for_scenario(df, var1_name, var2_name, units="TAF", tolerance=1e-6)
        if var_s == "S_MLRTN_":
            p_gte = probability_var1_gte_const_for_scenario(df, var1_name, S_MLRTN_level5constant, units="TAF")

        gte_col = f"Prob_{var_s}GTE_{var_lvl5D}"
        tier_col = f"{var_s}FloodTier"

        probs_row_data[gte_col] = p_gte
        tiers_row_data[tier_col] = classify_flood_tier(p_gte, tier1_tier2_thresh, tier2_tier3_thresh)

    probs_result_list.append(probs_row_data)
    tiers_result_list.append(tiers_row_data)

df_probs = pd.DataFrame(probs_result_list)
df_probs.set_index("Scenario", inplace=True)
df_tiers = pd.DataFrame(tiers_result_list)
df_tiers.set_index("Scenario", inplace=True)

print("Probabilities:")
display(df_probs.head())
print("\nTiers:")
display(df_tiers.head())

## Save Outputs

In [ ]:
df_probs.to_csv(probs_output_path)
df_tiers.to_csv(tiers_output_path)
print(f"Probabilities saved to: {probs_output_path}")
print(f"Tiers saved to: {tiers_output_path}")

## Merge All Metrics

In [ ]:
# Combine probabilities and tiers into single DataFrame
df_combined = pd.concat([df_probs, df_tiers], axis=1)
df_combined.index.name = "Scenario"

# Save combined output
combined_output_path = os.path.join(probs_output_dir, "floodrisk_all_metrics.csv")
df_combined.to_csv(combined_output_path)
print(f"Combined metrics saved to: {combined_output_path}")
display(df_combined.head())

In [ ]:
print("Done!")